In [118]:
import pickle
with open('dpp_error.pk', 'rb') as f:
    dpp_error = pickle.load(f)
    
from dppy.finite_dpps import FiniteDPP
import torch
    

In [257]:
(vec_matrix, encs, probs, tactics, state, theorem, temperature) = dpp_error


In [537]:
temperature = 1
scale = 1

logprobs = [t[1] / temperature for t in tactics]

probs = torch.softmax(torch.tensor(logprobs), dim=0) * scale

sim_matrix_ = torch.cat(encs[:16], dim=0)


sim_matrix = sim_matrix_ @ sim_matrix_.t()
sim_matrix = sim_matrix.cpu().numpy()


vec_matrix = torch.mul(sim_matrix_.cpu(), probs[:16].unsqueeze(1)).numpy()
vec_matrix = vec_matrix @ vec_matrix.T


In [538]:

DPP = FiniteDPP('likelihood', **{'L': sim_matrix})
# DPP = FiniteDPP('likelihood', **{'L': vec_matrix})
DPP.compute_K()

k_sum = sum(DPP.K_eig_vals)
k_sum


K (correlation) kernel computed via:
- eigendecomposition of L
- eig_K = eig_L/(1+eig_L)
- U diag(eig_K) U.T


5.550129127688706

In [539]:

DPP.sample_exact_k_dpp(size=10, mode='KuTa12')#, rng
# DPP.sample_exact()#, rng

len(DPP.list_of_samples[0])

10

In [540]:
[(i,t[0]) for i,t in enumerate(tactics[:16])]

[(0, 'refl'),
 (1, 'simp [t_ih]'),
 (2, 'cases tl'),
 (3, 'simp [<a>first_order.language.term.constants_to_vars</a>]'),
 (4, 'cases <a>first_order.language.term.func</a> ih t_ts'),
 (5, 'simp'),
 (6, 'rw [<a>first_order.language.term.constants_to_vars</a>, t_ih]'),
 (7, 'cases ih'),
 (8, 'rw <a>first_order.language.term.constants_to_vars</a>'),
 (9, 'rw t_ih'),
 (10, 'apply t_ih'),
 (11, 'simp [t_ih, t_ih]'),
 (12, 'rw [<a>first_order.language.term.constants_to_vars</a>]'),
 (13, 'cases ih : ih'),
 (14, 'simp [<a>first_order.language.term.func</a>]'),
 (15, 'unfold constants_to_vars')]

In [541]:
filtered = [tactics[i][0] for i in sorted(DPP.list_of_samples[0])]
filtered

['refl',
 'simp [<a>first_order.language.term.constants_to_vars</a>]',
 'cases <a>first_order.language.term.func</a> ih t_ts',
 'simp',
 'cases ih',
 'rw t_ih',
 'apply t_ih',
 'rw [<a>first_order.language.term.constants_to_vars</a>]',
 'simp [<a>first_order.language.term.func</a>]',
 'unfold constants_to_vars']

In [542]:
[t[0] for t in tactics[:16] if t[0] not in filtered]

['simp [t_ih]',
 'cases tl',
 'rw [<a>first_order.language.term.constants_to_vars</a>, t_ih]',
 'rw <a>first_order.language.term.constants_to_vars</a>',
 'simp [t_ih, t_ih]',
 'cases ih : ih']

In [536]:
sim_matrix

array([[ 1.0000000e+00,  1.8661527e-01,  4.4487622e-02,  2.0611839e-01,
         1.5896383e-01,  2.4703951e-01,  1.5352973e-01,  1.3078630e-01,
         1.5396339e-01,  1.2823169e-01,  1.1941235e-01,  1.4027569e-01,
         1.5180734e-01,  2.3152769e-02,  3.0760676e-01,  1.2645020e-01],
       [ 1.8661527e-01,  9.9999982e-01, -4.0923312e-02,  5.9728658e-01,
         1.5486634e-01,  8.3244592e-01,  3.3596334e-01,  7.0164613e-02,
         2.6825100e-01,  4.6864781e-01,  7.5878508e-02,  9.5124686e-01,
         2.8019714e-01,  3.3359282e-02,  6.2359959e-01,  1.5632653e-01],
       [ 4.4487622e-02, -4.0923312e-02,  9.9999988e-01,  1.0914179e-02,
         2.7858400e-01, -1.1494008e-02,  1.7513575e-01,  3.7672874e-01,
         1.3215598e-01,  1.6931280e-01,  1.5996693e-01, -2.1129776e-02,
         1.3106301e-01,  2.6291674e-01, -3.1569661e-03,  1.2575918e-01],
       [ 2.0611839e-01,  5.9728658e-01,  1.0914179e-02,  1.0000000e+00,
         3.3717984e-01,  6.0191965e-01,  7.5446385e-01,  5.83

In [303]:
1e1     

10.0

In [1]:
import glob

bestfs_traces = glob.glob('../runs/bestfs-novel-2/*')
diversity_traces = glob.glob('../runs/search/diversity-temp=1/2024_07_22/13_33_28/traces/0/*')

In [2]:
names = { d.split('/')[-1] for d in bestfs_traces }
diversity_traces = [d for d in diversity_traces if d.split('/')[-1] in names]
names = { d.split('/')[-1] for d in diversity_traces }
bestfs_traces = [d for d in bestfs_traces if d.split('/')[-1] in names]
                    

In [3]:
len(bestfs_traces)

268

In [7]:
from tqdm import tqdm
import pickle

successes = []
fails = []
for i in tqdm(range(len(bestfs_traces))):
    try:
        with open(bestfs_traces[i], 'rb') as f:
            bestfs = pickle.load(f)
        with open(diversity_traces[i], 'rb') as f:
            diversity = pickle.load(f)
    except:
        continue
        
    if bestfs.proof and not diversity.proof:
        successes.append((bestfs, diversity))
    elif not bestfs.proof and diversity.proof:
        fails.append((bestfs, diversity))
        
        
        

100%|██████████| 268/268 [01:45<00:00,  2.55it/s]


In [8]:
len(successes)

18

In [9]:
len(fails)

7

In [57]:
# [s[0].proof for s in successes]

In [58]:
# [s[1].proof for s in fails]

In [104]:
best_trace, diverse_trace = successes[4]


In [105]:
best_trace.proof

['refine <a>inner_product_space.ext_inner_left_basis</a> b (λ i, _)',
 'rw ←<a>inner_conj_symm</a>',
 'rw [h i, <a>inner_conj_symm</a>]']

In [114]:
len([d for d in best_trace.tree.out_edges if not isinstance(d.dst[0], ErrorNode)])


12

In [116]:
len([d for d in diverse_trace.tree.out_edges if not isinstance(d.dst[0], ErrorNode)]), len([d for d in best_trace.tree.out_edges if not isinstance(d.dst[0], ErrorNode)])



(3, 12)

In [107]:
[(d,d.dst[0]) for d in diverse_trace.tree.out_edges ]

[(Edge(tactic='ext i', tac_logprob=-1.7376303672790527, goal_logprob=0.0, time=0.0369220589636825),
  ErrorNode(inner=EnvironmentError(message='gen_tac_and_capture_res_failed: pos=(some ⟨1, 2⟩) msg="only constants and Pi types are supported: E" tactic_state="𝕜 : Type u_1,\nE : Type u_2,\n_inst_1 : is_R_or_C 𝕜,\n_inst_2 : normed_add_comm_group E,\n_inst_3 : inner_product_space 𝕜 E,\nι : Type u_3,\nx y : E,\nb : basis ι 𝕜 E,\nh : ∀ (i : ι), inner x (⇑b i) = inner y (⇑b i)\n⊢ x = y"'))),
 (Edge(tactic='classical', tac_logprob=-3.023669958114624, goal_logprob=0.0, time=0.06275517103495076),
  ErrorNode(inner=TreeError(message='Selected goal remains in response'))),
 (Edge(tactic='ext', tac_logprob=-3.1678192615509033, goal_logprob=0.0, time=0.08400630694814026),
  ErrorNode(inner=EnvironmentError(message='gen_tac_and_capture_res_failed: pos=(some ⟨1, 2⟩) msg="only constants and Pi types are supported: E" tactic_state="𝕜 : Type u_1,\nE : Type u_2,\n_inst_1 : is_R_or_C 𝕜,\n_inst_2 : normed_a

In [108]:
from experiments.end_to_end.proof_node import ErrorNode

node1 = [d.dst for d in best_trace.tree.out_edges if d.tactic == best_trace.proof[0]][0][0]

In [109]:
node2 = [d.dst for d in node1.out_edges if d.tactic == best_trace.proof[1]]
node2[0][0]

InternalNode(goal='𝕜 : Type u_1,\nE : Type u_2,\n_inst_1 : is_R_or_C 𝕜,\n_inst_2 : normed_add_comm_group E,\n_inst_3 : inner_product_space 𝕜 E,\nι : Type u_3,\nx y : E,\nb : basis ι 𝕜 E,\nh : ∀ (i : ι), inner x (⇑b i) = inner y (⇑b i),\ni : ι\n⊢ ⇑conj (inner x (⇑b i)) = inner (⇑b i) y', _status=<Status.PROVED: 'Proved'>)

In [110]:
diverse1  = [d.dst[0] for d in diverse_trace.tree.out_edges if d.dst[0 ] == node1][0]


IndexError: list index out of range

In [55]:
diverse2 = [d.dst for d in diverse1.out_edges if d.dst[0] == node2[0][0]]

In [56]:
diverse2

[]

In [ ]:
# bestfs success 0, missed out on state by removing simp tactic with relevant lemma (depth 1)
# success 1, missed out on proof at first level by removing rw tactic with relevant lemma (depth 0)
# success 2, missed out on proof at first level by removing rw tactic with relevant lemma (depth 0). Other tactics with same lemma, however they resulted in an error.
# success 3, missed tactic but had lemma (simp_rw<- vs simp_rw, timed out)
# success 4, 